# CSC14120 Parallel Programming: Final Project, Midterm
## Panorama stitching: proposal and sequential version (V0)

**Group 06**. Dataset: [link to the team's Google Drive folder](https://drive.google.com/file/d/19f2lNy37TMzK08LO_i0CzrAccBBzpgAc/view?usp=sharing)

How to run: **Runtime → Change runtime type → GPU**, set `DRIVE_DIR` in the setup cell, then **Runtime → Run all**.

## 3.1 Team information and work plan

| Member | Student ID | Responsible for |
|---|---|---|
| Quách Châu Hạo Kiệt | 23127178 | Stages 1-5 (preprocessing, RANSAC homography) in every version; dataset capture |
| Phan Thanh Bình | 23127028 | Stages 3–4 (ORB description, matching) in every version; profiling and charts |
| Tôn Thất Minh Đăng | 23127037 | Stages 2–6 (keypoints, warping and blending) in every version; CLI, timing harness, notebook |

<!-- | Version | Stages 1–5 | Stages 3–4 | Stages 2–6 | Target |
|---|---|---|---|---|
| V0 sequential | Quách Châu Hạo Kiệt | Phan Thanh Bình | C | Midterm |
| V1 basic CUDA | Quách Châu Hạo Kiệt | B | C | Final, week 4 |
| V2, V3 optimized | chosen from the V1 profile | | | Final, week 5 |
| V4 (+V5) optimized | chosen from the V3 profile | | | Final, week 6 | -->
Work Plan:

<table>
  <thead>
    <tr>
      <th>Version</th>
      <th>Stages 1 &amp; 5</th>
      <th>Stages 3–4</th>
      <th>Stages 2 &amp; 6</th>
      <th>Target</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>V0 sequential</b></td>
      <td rowspan="4" align="center"><b>Quách Châu Hạo Kiệt</b></td>
      <td rowspan="4" align="center"><b>Phan Thanh Bình</b></td>
      <td rowspan="4" align="center"><b>Tôn Thất Minh Đăng</b></td>
      <td>Midterm</td>
    </tr>
    <tr>
      <td><b>V1 basic CUDA</b></td>
      <td>Final, week 4</td>
    </tr>
    <tr>
      <td><b>V2, V3 optimized</b></td>
      <td>Final, week 5</td>
    </tr>
    <tr>
      <td><b>V4 (+V5) optimized</b></td>
      <td>Final, week 6</td>
    </tr>
  </tbody>
</table>

## 3.2 Problem understanding

We are given two overlapping photos taken by **rotating the camera in place**. The goal attaching both images together to form one wide image where the shared part lines up exactly.

Because the camera only rotates, every pixel of the right image B maps to the left image A through a theoretical pipeline: Assume an object N in space projecting into image B as a pixel. We reverse map (unproject) this pixel from image B's 2D plane back to its 3D light ray in real-world 3D coordinate system, then rotate this light ray into image A's orientation (i.e project another ray of light from N onto image A's camera sensor), and finally this light ray would leave a 2D projection on image A's plane which would be the corresponding pixel in A that represent object N. This 3 step process can be illustrated mathematically as follows:

1. **Unprojection ($K_B^{-1}$):** Inverts the camera intrinsic calibration matrix $K_B$, transforming 2D sensor coordinates $\mathbf{x}_B$ back into a 3D directional vector in space:
   $$\mathbf{v} = K_B^{-1} \mathbf{x}_B$$
Where K is the matrix with intrinsic information about the camera in question, with the form:
    $$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ \mathbf{0} & \mathbf{0} & \mathbf{1} \end{bmatrix}$$
with $(f_x, f_y)$ being the focal length in pixels and $(c_x, c_y)$ being the optical center of the camera. 

2. **3D Rotation ($R$):** Rotates the physical ray into the orientation of Camera A:
   $$\mathbf{v}' = R \mathbf{v} = R K_B^{-1} \mathbf{x}_B$$

3. **Projection ($K_A$):** Projects the rotated ray onto Camera A's image sensor:
   $$\mathbf{x}_A \sim K_A \mathbf{v}' = \underbrace{(K_A R K_B^{-1})}_{H} \mathbf{x}_B$$

Thus, all three transformation can be grouped into a single 3×3 matrix **H** (a homography). In homogeneous coordinates, a pixel $(x, y)$ from Image B is transformed by:
$$\begin{bmatrix} x' \\ y' \\ w' \end{bmatrix} = H \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} = \begin{bmatrix} H_{0} & H_{1} & H_{2} \\ H_{3} & H_{4} & H_{5} \\ H_{6} & H_{7} & H_{8} \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$
Because projection operates up to a scale factor, $w'$ represents the projective depth. To map back to 2D pixel coordinates $(x_A, y_A)$ on Image A, we perform **perspective division** (finding the intersection between the projection line and image A's plane, also replicating the close-far effect in 2D):
$$x_A = \frac{x'}{w'} = \frac{H_{0}x + H_{1}y + H_{2}}{H_{6}x + H_{7}y + H_{8}}, \qquad y_A = \frac{y'}{w'} = \frac{H_{3}x + H_{4}y + H_{5}}{H_{6}x + H_{7}y + H_{8}}$$
This is exactly how the host function `applyH()` in `src/common.h` and the inverse warping kernel in Stage 6 compute pixel coordinates. Everything in the pipeline exists to estimate this matrix $H$ reliably without needing to know the camera's physical parameters ($K$ or $R$) beforehand.

1. **Preprocessing:** convert to grayscale and blur, so noise does not create fake corners.
2. **Keypoint detection:** find a few thousand distinctive points (corners) that can be found again in the other photo.
3. **Description:** give each keypoint a compact fingerprint of its neighbourhood.
4. **Matching:** pair each keypoint in A with the most similar one in B and drop ambiguous pairs.
5. **Homography estimation:** some pairs are still wrong, so RANSAC repeatedly fits H to 4 random pairs and keeps the H most pairs agree with.
6. **Warping and blending:** move B into A's frame using H and blend the overlap into one image.

Stages 1–3 run on each image independently; stages 4–6 need both.

## 3.3 Chosen methods

| Stage | Method | Main parameters | Quality | GPU suitability |
|---|---|---|---|---|
| 1 | Grayscale + separable Gaussian blur | σ = 2/3 (r/3 with r=2), 5×5 kernel, clamped border | Removes sensor noise without artifacts | Stencil, one thread per pixel; separable (2k instead of k² work); shared-memory tiles later |
| 2 | Harris response + 3×3 non-max suppression + top-K | k = 0.04, 5×5 window, K = 2000, 16 px border | Stable corners under small rotation; fixed K keeps matching cost predictable | Uniform per-pixel work (no divergence); compaction and top-K are classic parallel patterns |
| 3 | ORB: intensity-centroid orientation + rotated BRIEF | 31×31 patch, 256 tests, fixed pattern (seed 12345) | Handles small camera roll; enough for overlapping photos | One thread per keypoint; pattern fits in constant memory; 32-byte result |
| 4 | Brute-force Hamming nearest neighbour + Lowe ratio test + cross-check | ratio < 0.8, mutual best | Exact search; ratio test and cross-check remove ambiguous pairs | All-pairs like matrix multiply; XOR + `__popc`; tiling in shared memory |
| 5 | RANSAC + normalized 4-point DLT, refit on inliers | 1000 iterations, 3 px threshold, fixed samples (seed 2026) | Robust to wrong matches; refit improves accuracy | Independent hypotheses (one thread or warp each); inlier count is a reduction |
| 6 | Inverse warping + bilinear sampling + feathering | weights ∝ distance to the image border | No holes, smooth sampling, no hard seam | One independent gather per output pixel; texture memory later |

**Alternatives we rejected:** SIFT/DoG (multi-scale, float descriptors: heavier and not needed when the camera only rotates); FAST (branchy test causes warp divergence and has no ranking score); k-d tree or FLANN matching (approximate and branchy on the GPU); plain least squares for H (not robust to wrong matches); forward warping (holes and write conflicts); multi-band blending (more stages; only if feathering fails).

All versions share these parameters, the ORB pattern and the RANSAC samples (`src/common.h`), so their outputs can be compared directly.

## Setup: copy `src/` and build

The notebook builds the program from the submitted `src/` folder, nothing is copied into notebook cells. Put `src/`, `tools/`, `Makefile` and `dataset/` (and optionally `raw/`) in the team's shared Drive folder.

In [ ]:
!nvidia-smi -L
!nvcc --version | tail -n 2

In [ ]:
import os, shutil

DRIVE_DIR = '/content/drive/MyDrive/CSC14120_Project'   # <- the team's shared folder
WORK = '/content/project'

from google.colab import drive
drive.mount('/content/drive')
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(DRIVE_DIR, WORK, ignore=shutil.ignore_patterns('out', 'stitch', '*.exe', '*.ipynb'))
os.chdir(WORK)
print(sorted(os.listdir('src')))

In [ ]:
!make clean && make

In [ ]:
import os, subprocess, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

STAGES = ['1 preprocess', '2 detect', '3 describe', '4 match', '5 homography', '6 warp+blend']
CSV_COLS = (['version', 'width', 'height', 'runs'] + STAGES +
            ['h2d', 'd2h', 'total', 'min_total', 'kp_a', 'kp_b', 'matches', 'inliers', 'gt_err', 'ref_err'])
SIZES = ['small', 'medium', 'large']
os.makedirs('out', exist_ok=True)

def stitch(left, right, out, version=0, runs=5, extra=(), show=False):
    # Runs ./stitch and returns its CSV line as a dict.
    cmd = ['./stitch', left, right, out, '--version', str(version), '--runs', str(runs), *extra]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if show or res.returncode != 0:
        print(res.stdout, res.stderr)
    res.check_returncode()
    line = [l for l in res.stdout.splitlines() if l.startswith('CSV,')][-1]
    return dict(zip(CSV_COLS, [float(v) for v in line.split(',')[1:]]))

def show_images(paths, titles, cols=2, width=16):
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(width, width / cols * 0.6 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, p, t in zip(axes.flat, paths, titles):
        ax.imshow(Image.open(p))
        ax.set_title(t)
    plt.tight_layout()
    plt.show()

## 3.6 Dataset

Each site has one standard pair; there is one challenging pair for failure analysis. Raw photos (`raw/<site>/left.jpg`, `right.jpg`) are cropped to 16:9 and resized to 1280×720, 1920×1080 and 3840×2160 by `tools/prepare_dataset.py`.

| Site | Type | Location | Notes |
|---|---|---|---|
| site1 | indoor | Three O'Clock Coffee Shop, 771 Tran Hung Dao St. | |
| site2 | outdoor | Balcony Floor 8, HCMUS | |
| site3 | indoor | Nép Coffee Shop, 63C/2 Tran Dinh Xu St. | |
| challenge | outdoor + challenging | War Remnants Museum, 28 Vo Van Tan St. | low texture (blur) + moving people |

The synthetic pair with its ground-truth homography comes from the course starter notebook. It is stored as `dataset/synthetic/<small|medium|large>/left.*`, `right.*` and `H_gt.txt` (3×3, maps B to A).

In [ ]:
if os.path.isdir('raw'):
    !python tools/prepare_dataset.py raw dataset

SITES = sorted(d for d in os.listdir('dataset') if os.path.isdir(f'dataset/{d}') and d != 'synthetic')
print('sites:', SITES)
paths, titles = [], []
for s in SITES:
    for side in ('left', 'right'):
        paths.append(glob.glob(f'dataset/{s}/medium/{side}.*')[0])
        titles.append(f'{s} / {side}')
show_images(paths, titles)

## 3.4 Sequential version (V0)

`src/v0_sequential.cpp`, single-threaded C++ (no OpenMP, threads or hand-written SIMD), compiled with `-O3`. The interface every version implements is in `src/versions.h`:

```cpp
void runV0(const ImageU8& A, const ImageU8& B, const Params& params, Result& result, Timing& timing);
```

Design points:
- **Deterministic:** non-max suppression and top-K break ties by pixel index; the ORB pattern and RANSAC samples come from a fixed-seed generator shared with the GPU versions (`src/common.h`).
- **Same precision as the GPU:** per-pixel work uses `float`; only the small 8×8 DLT solve uses `double`.
- **Timing:** `std::chrono::high_resolution_clock` around each stage; one warm-up run, then the average of `--runs` runs (default 5). Image loading and saving are not timed.

## 3.5 V0 results and profiling

### Correctness on the synthetic pair
The estimated H is compared with the ground truth by mapping the four corners of B with both and averaging the distance (`--gt`).

In [ ]:
rows = []
for size in SIZES:
    d = f'dataset/synthetic/{size}'
    if not os.path.exists(f'{d}/H_gt.txt'):
        print(f'{d}: synthetic pair not found, copy it from the starter notebook')
        continue
    left, right = glob.glob(f'{d}/left.*')[0], glob.glob(f'{d}/right.*')[0]
    r = stitch(left, right, f'out/synthetic_{size}.png', runs=1,
               extra=['--gt', f'{d}/H_gt.txt'], show=(size == 'medium'))
    rows.append({'size': size, 'resolution': f"{int(r['width'])}x{int(r['height'])}", 'matches': int(r['matches']),
                 'inliers': int(r['inliers']), 'corner error (px)': round(r['gt_err'], 3)})
if rows:
    display(pd.DataFrame(rows))
    shown = [f'out/synthetic_{s}.png' for s in SIZES if os.path.exists(f'out/synthetic_{s}.png')]
    show_images(shown[-1:], ['synthetic pair: V0 panorama'], cols=1, width=14)

### Panoramas: one per site (medium size)

In [ ]:
paths, titles = [], []
for s in SITES:
    left, right = glob.glob(f'dataset/{s}/medium/left.*')[0], glob.glob(f'dataset/{s}/medium/right.*')[0]
    r = stitch(left, right, f'out/{s}_medium.jpg', runs=1, extra=['--save-h', f'out/{s}_medium_H.txt'])
    paths.append(f'out/{s}_medium.jpg')
    titles.append(f"{s}: {int(r['matches'])} matches, {int(r['inliers'])} inliers")
show_images(paths, titles, cols=1, width=14)

### Per-stage timing for all three resolutions
Average of 5 timed runs after one warm-up, per site; the tables below average over the standard sites.

In [ ]:
timing = []
for s in SITES:
    if s.startswith('challeng'):
        continue                      # the challenging pair is for failure analysis, not the benchmark
    for size in SIZES:
        left, right = glob.glob(f'dataset/{s}/{size}/left.*')[0], glob.glob(f'dataset/{s}/{size}/right.*')[0]
        r = stitch(left, right, f'out/{s}_{size}.jpg', runs=5)
        r.update(site=s, size=size)
        timing.append(r)
timing = pd.DataFrame(timing)
!nvidia-smi --query-gpu=name --format=csv,noheader

v0 = timing.groupby('size', sort=False)[STAGES + ['total', 'min_total']].mean().reindex(SIZES)
print('V0 time per stage (ms), averaged over sites')
display(v0.round(1))
share = v0[STAGES].div(v0['total'], axis=0) * 100
print('Share of end-to-end time (%)')
display(share.round(1))

ax = v0[STAGES].plot.bar(stacked=True, figsize=(9, 5), rot=0)
ax.set_ylabel('time (ms)')
ax.set_title('V0 time per stage by resolution')
plt.tight_layout()
plt.show()

### Hot stages

In [ ]:
for size in SIZES:
    top = share.loc[size].sort_values(ascending=False)
    print(f"{size:>6}: " + ', '.join(f'{st} {pct:.1f}%' for st, pct in top.items()))
growth = v0.loc['large', STAGES] / v0.loc['small', STAGES]
print('\nGrowth from small to large (x9 pixels):')
print(growth.round(1).to_string())

*Write the interpretation here, quoting only numbers from the outputs above: which stages dominate, and which grow with the pixel count (stages 1, 2, 6) versus the fixed keypoint count (stages 3, 4, 5).*

### Amdahl estimate
If the stages we move to the GPU take a fraction *p* of V0's end-to-end time, the overall speedup can never exceed 1 / (1 − p).

In [ ]:
GPU_STAGES = STAGES          # we plan to move all six stages to the GPU
p = v0[GPU_STAGES].sum(axis=1) / v0['total']
amdahl = pd.DataFrame({'p (accelerated fraction)': p.round(4), 'max speedup 1/(1-p)': (1 / (1 - p)).round(1)})
display(amdahl)

The time outside the six stages (allocation, building the ORB pattern and RANSAC samples) bounds the speedup. In the GPU versions, host↔device copies are added on top, so we will also report them separately.

## 3.7 GPU plan

**Stages moved to the GPU:** all six. The profile above shows where the time goes; the stages that scale with the pixel count are the main targets.

| Stage | V1 thread mapping | Notes |
|---|---|---|
| 1 Grayscale + blur | 1 thread per pixel; blur as a row pass and a column pass | global memory only in V1 |
| 2 Harris + NMS + top-K | 1 thread per pixel for gradients, window sums, response and NMS; survivors appended with a global atomic counter; top-K by sorting candidates on the device | variable keypoint count is the hard part |
| 3 ORB | 1 thread per keypoint: orientation, then 256 tests | pattern copied once to the device |
| 4 Matching | 1 thread per keypoint of A scans all of B (best, second best); a second kernel for B → A cross-check | 2000 × 2000 Hamming distances |
| 5 RANSAC | 1 thread per hypothesis: 4-point DLT + inlier count; argmax on the device; refit on inliers | uses the same pre-generated samples as V0 |
| 6 Warp + blend | 1 thread per output pixel | canvas size from the 4 warped corners |

**Optimizations we expect to try (each version adds one, chosen from profiling):**
1. Shared-memory tiling with halos for the blur and Harris stencils, filter weights in constant memory (stages 1–2).
2. Tiled matching: descriptors of B staged in shared memory, `__popc` on 32-bit words (stage 4).
3. Prefix-sum compaction instead of a global atomic for keypoints, plus CUDA streams so images A and B run stages 1–3 concurrently.

**Main risks:** GPU results differing slightly from V0 (float rounding changes keypoint ranking at ties). We will compare keypoint overlap, matches and the corner error of H rather than exact equality. Also: top-K selection on the GPU, RANSAC's 8×8 solve per thread (register pressure, double precision on Colab's T4), and the challenging pair failing to match.

## References

1. S. Nayar, "Image Stitching", First Principles of Computer Vision, monograph FPCV-2-4, 2025.
2. R. Szeliski, *Computer Vision: Algorithms and Applications*, 2nd ed., Springer, 2022 (feature detection and matching, image alignment and stitching).
3. C. Harris and M. Stephens, "A Combined Corner and Edge Detector", Alvey Vision Conference, 1988.
4. E. Rublee, V. Rabaud, K. Konolige, G. Bradski, "ORB: An efficient alternative to SIFT or SURF", ICCV 2011.
5. D. Lowe, "Distinctive Image Features from Scale-Invariant Keypoints", IJCV 2004 (ratio test).
6. M. Fischler and R. Bolles, "Random Sample Consensus", Communications of the ACM, 1981.
7. R. Hartley and A. Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed., 2004 (normalized DLT).
8. S. Barrett, stb_image v2.30 and stb_image_write v1.16, public domain, https://github.com/nothings/stb
9. CSC14120 homework templates (`CHECK` macro and `GpuTimer` in `src/cuda_utils.cuh`).

### AI tool usage declaration
*(Edit to match what the team actually did.)* We used Claude (Anthropic) to help plan the project, explain the methods, and draft the initial code skeleton (`src/`, `tools/`) and this notebook's structure. Every member reviewed the code, and we can explain and modify every part of it.